# 06 — RAG Chatbot (Retrieval-Augmented Generation)
**Objectif :** un assistant qui répond aux questions sur le modèle, les variables et la méthodologie, en s'appuyant sur la documentation du projet.

**Architecture RAG :**
1. **Indexation** : la base de connaissances (data dictionary + model card + méthodologie) est découpée en chunks, chaque chunk est encodé en vecteur par `sentence-transformers`.
2. **Stockage** : les vecteurs sont indexés dans **FAISS** (recherche par similarité rapide).
3. **Retrieval** : à chaque question, on encode la question et on récupère les *k* chunks les plus proches.
4. **Génération** : on passe ces chunks comme contexte à **Claude**, qui rédige la réponse.

**Pourquoi RAG plutôt que tout mettre dans le prompt ?** Le retrieval permet de ne donner au LLM que les passages pertinents — ça scale à de grandes bases documentaires et limite les hallucinations en ancrant la réponse dans des sources.

**LLM :** Claude Haiku 4.5 (`claude-haiku-4-5`) — rapide et économique, suffisant pour de la Q&A documentaire.

## 0. Installation et clé API

La clé API Anthropic doit être dans un fichier `.env` à la racine du projet :
```
ANTHROPIC_API_KEY=sk-ant-...
```
Le `.env` est déjà dans le `.gitignore` (la clé ne sera jamais commitée).

In [1]:
import os
import glob
import pickle
import numpy as np
from dotenv import load_dotenv

load_dotenv(os.path.join('..', '.env'))

raw_key = os.getenv('ANTHROPIC_API_KEY', '')
# On considère la clé valide seulement si elle est renseignée et n'est pas le placeholder
api_key = raw_key if raw_key.startswith('sk-ant-') and 'remplace' not in raw_key else None

if api_key:
    print('Clé API Anthropic chargée — génération Claude active.')
else:
    print('Pas de clé API valide → mode local (retrieval seul, sans génération LLM).')
    print('Pour activer Claude : mets ta clé dans .env (ANTHROPIC_API_KEY=sk-ant-...) et relance.')

Pas de clé API valide → mode local (retrieval seul, sans génération LLM).
Pour activer Claude : mets ta clé dans .env (ANTHROPIC_API_KEY=sk-ant-...) et relance.


## 1. Chargement et chunking de la base de connaissances

On lit les fichiers markdown de `knowledge_base/` et on les découpe en chunks par section (titres `##`).
Découper par section garde des unités sémantiquement cohérentes : une section = un sujet.

In [ ]:
import sys
sys.path.insert(0, os.path.join('..', 'src'))  # accès au package credit_risk
from credit_risk.rag import chunk_markdown, retrieve, answer, has_api_key

KB_DIR = os.path.join('..', 'knowledge_base')
all_chunks = []
for path in sorted(glob.glob(os.path.join(KB_DIR, '*.md'))):
    source = os.path.basename(path)
    with open(path, encoding='utf-8') as f:
        all_chunks.extend(chunk_markdown(f.read(), source))

print(f"{len(all_chunks)} chunks extraits de {len(glob.glob(os.path.join(KB_DIR, '*.md')))} documents")
print(f"\nExemple de chunk :")
print(all_chunks[3]['text'][:300])

## 2. Embeddings + index FAISS

On encode chaque chunk avec `all-MiniLM-L6-v2` (modèle léger, 384 dimensions, tourne en local sans GPU).
Les vecteurs sont normalisés (norme L2 = 1) pour que le produit scalaire FAISS équivaille à une similarité cosinus.

In [3]:
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer('all-MiniLM-L6-v2')

texts = [c['text'] for c in all_chunks]
embeddings = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True)
embeddings = np.asarray(embeddings, dtype='float32')

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # Inner Product sur vecteurs normalisés = cosinus
index.add(embeddings)

print(f"Index FAISS construit : {index.ntotal} vecteurs de dimension {dim}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 2/2 [00:01<00:00,  1.06it/s]

Index FAISS construit : 51 vecteurs de dimension 384


## 3. Fonction de retrieval

In [ ]:
# retrieve() est importé de src.rag — signature (question, index, chunks, embedder, k)
hits = retrieve("Pourquoi avez-vous retiré le code postal du modèle ?", index, all_chunks, embedder, k=3)
for h in hits:
    print(f"[{h['score']:.3f}] {h['source']} — {h['title']}")

## 4. Génération avec Claude

La fonction `generate()` prend le contexte récupéré + la question, et demande à Claude de répondre **uniquement** à partir du contexte.

**Prompt caching :** le system prompt (instructions, stables d'une question à l'autre) porte un `cache_control`. Les requêtes suivantes réutilisent ce préfixe mis en cache (~0.1× le prix). Le contexte récupéré et la question, eux, varient à chaque appel et sont placés dans le message utilisateur (après le préfixe caché).

In [ ]:
# generate() : wrapper local qui enchaîne retrieve (src.rag) + answer (src.rag)
# answer() gère le prompt caching et bascule en mode local si pas de clé API.

def generate(question: str, k: int = 4) -> str:
    hits = retrieve(question, index, all_chunks, embedder, k=k)
    text, context = answer(question, hits)
    sources = ', '.join(sorted(set(h['title'] for h in hits)))
    if text is None:
        return f"[Mode local — passages pertinents | sources : {sources}]\n\n{context}"
    return f"[sources : {sources}]\n\n{text}"

print('Mode :', 'Claude actif' if has_api_key() else 'local (retrieval seul)')

## 5. Démonstration

In [6]:
print(generate("Quelle est l'AUC du modèle et quel modèle est retenu en production ?"))

[Mode sans LLM — passages les plus pertinents]

[02_model_card.md — Régression logistique (modèle retenu pour la production)]
- AUC ROC sur le test : 0.74.
- Avantages : interprétable, conforme aux exigences réglementaires (Bâle III, guidelines EBA), rapide.
- C'est le modèle recommandé pour un déploiement réel en banque.

---

[02_model_card.md — Limites et précautions]
- Le modèle ne capture pas les chocs macroéconomiques postérieurs à l'octroi.
- Le target encoding géographique doit être revalidé dans le temps (stabilité des taux de défaut par zone).
- Pour une nouvelle zone géographique sans historique, l'encodage retombe sur la moyenne globale.
- Le modèle est entraîné sur des prêts conformes Freddie Mac ; il ne couvre pas les prêts jumbo ou non-conformes.

---

[01_data_dictionary.md — fico_dti_interaction]
Produit (850 - credit_score) × dti / 100. Capture les profils cumulant un FICO faible et un DTI élevé. C'est le driver #1 du modèle selon l'analyse SHAP.

---

[03_methodology

In [7]:
print(generate("Pourquoi avez-vous retiré le code postal des features ?"))

[Mode sans LLM — passages les plus pertinents]

[02_model_card.md — Décision de feature engineering importante]
La variable postal_code (code postal) a été retirée du modèle. Bien qu'identifiée comme driver #1 par SHAP dans une version antérieure, la retirer a fait *monter* l'AUC (de 0.72 à 0.735 sur XGBoost). Explication : sa cardinalité très élevée (milliers de zones, beaucoup avec moins de 10 prêts) faisait que le target encoding mémorisait le train sans généraliser — un overfit géographique. msa et property_state, plus grossiers, sont plus stables et ont été conservés. Leçon : l'importance d'une feature (SHAP) n'égale pas son utilité prédictive ; seule la validation sur données vierges tranche.

---

[03_methodology.md — Choix méthodologiques clés à retenir]
1. Split avant encodage pour éviter le leakage.
2. Retrait des variables de performance (leakage de la cible).
3. Retrait de postal_code (overfit géographique, valide par la hausse d'AUC).
4. Régression logistique retenue en pr

In [8]:
print(generate("Que signifie un DTI supérieur à 43 % et pourquoi c'est important ?"))

[Mode sans LLM — passages les plus pertinents]

[01_data_dictionary.md — dti (Debt-to-Income ratio)]
Ratio dette/revenu de l'emprunteur, en pourcentage. Mesure la part du revenu mensuel consacrée au remboursement de toutes les dettes. Un DTI supérieur à 43% dépasse le seuil "Qualified Mortgage" (QM) défini par le CFPB et signale un risque accru. Valeur sentinelle 999 = manquant.

---

[01_data_dictionary.md — rate_spread]
Écart entre le taux du prêt et le taux moyen de son millésime d'origination. Proxy du scoring interne appliqué par le vendeur du prêt.

---

[01_data_dictionary.md — oltv (Original Loan-to-Value)]
Ratio prêt/valeur du bien au moment de l'octroi, en pourcentage. Mesure l'apport personnel : un OLTV de 80 signifie que le prêt couvre 80% de la valeur du bien (20% d'apport). Au-delà de 80%, une assurance hypothécaire (MIP) est généralement obligatoire. Valeur sentinelle 999 = manquant.

---

[01_data_dictionary.md — risk_count]
Somme des trois flags ci-dessus (0 à 3). Le t

In [9]:
# Deuxième appel rapproché → le system prompt doit être lu depuis le cache (cache_read > 0)
print(generate("Comment la cible 'default' est-elle construite ?"))

[Mode sans LLM — passages les plus pertinents]

[03_methodology.md — Étape 2 — Construction de la cible]
Les données de performance (une ligne par prêt par mois) sont agrégées en une ligne par prêt : delinquance maximale, nombre de mois en retard, code de solde final. La cible default est dérivée de ces agrégats. Les variables de performance sont ensuite retirées des features car elles constituent du data leakage (elles servent à définir la cible et ne sont pas connues à l'octroi).

---

[01_data_dictionary.md — default]
Variable binaire construite pour la modélisation. default = 1 si le prêt a atteint 90+ jours de retard (delinquency >= 3) OU a fini en saisie/REO (zero_balance_code dans {03, 09}). Sinon default = 0. Taux de défaut observé dans l'échantillon : 5.57%.

---

[01_data_dictionary.md — original_loan_term]
Durée du prêt en mois. Les valeurs standards sont 360 (30 ans), 180 (15 ans), 240 (20 ans).

---

[01_data_dictionary.md — first_time_homebuyer_flag]
Indicateur primo-accé

## 6. Sauvegarde de l'index pour l'app Streamlit

On persiste l'index FAISS et les chunks pour que l'app Streamlit n'ait pas à les recalculer à chaque démarrage.

In [10]:
MODELS_DIR = os.path.join('..', 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

faiss.write_index(index, os.path.join(MODELS_DIR, 'rag_index.faiss'))
with open(os.path.join(MODELS_DIR, 'rag_chunks.pkl'), 'wb') as f:
    pickle.dump(all_chunks, f)

print('Sauvegardé :')
print('  models/rag_index.faiss')
print('  models/rag_chunks.pkl')
print(f'\nIndex prêt : {index.ntotal} chunks indexés.')

Sauvegardé :
  models/rag_index.faiss
  models/rag_chunks.pkl

Index prêt : 51 chunks indexés.
